# 2Dify AI — FULL-2D Video Reconstruction Worker

**Goal:** one source video upload → automatic 2D guide → generative video-to-video → strict full-frame 2D reconstruction → original audio.

The phone only uploads the video. A browser color filter is NOT used as the claimed AI conversion.


## 1. Install and upload ONE video


In [ ]:
!pip -q install -U gradio_client huggingface_hub imageio-ffmpeg pillow opencv-python-headless
from pathlib import Path
import os, json, subprocess, cv2, numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from google.colab import files
from gradio_client import Client, handle_file
import imageio_ffmpeg
up=files.upload()
source_video=next(iter(up))
print('Source:',source_video)


## 2. Automatically create the 2D style guide
This guide is generated automatically from the first frame; the generative model does the actual visual reconstruction.


In [ ]:
ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()
raw='/content/2dify_first.png'
subprocess.run([ffmpeg,'-y','-i',source_video,'-frames:v','1','-vf','scale=768:-2',raw],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

# Automatic 2D illustration guide: posterization + edge linework + color smoothing.
# This is only the guide; ID-V2V performs the actual generative video reconstruction.
bgr=cv2.imread(raw)
bgr=cv2.bilateralFilter(bgr,9,55,55)
lab=cv2.cvtColor(bgr,cv2.COLOR_BGR2LAB)
l,a,b=cv2.split(lab)
l=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(l)
lab=cv2.merge([l,a,b])
bgr=cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)
rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
rgb=(rgb//32)*32
gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
edges=cv2.Canny(gray,70,150)
edges=cv2.dilate(edges,np.ones((2,2),np.uint8),iterations=1)
guide=rgb.copy()
guide[edges>0]=[25,25,25]
Image.fromarray(guide).save('/content/2d_style_guide.png')
display(Image.fromarray(guide))


## 3. Strict FULL-2D prompt


In [ ]:
STYLE_PROMPT='''FULL-FRAME 2D ANIMATED RECONSTRUCTION. Re-render EVERY visible region as coherent 2D illustrated artwork. Characters, faces, eyes, teeth, hair, skin, hands, clothes, accessories, animals, vehicles, furniture, tools, buildings, architecture, plants, sky, clouds, water, ground, shadows, highlights, reflections, particles and background MUST all be 2D illustrated shapes and linework. Use consistent hand-drawn/cartoon/anime/flat-illustration rendering across the entire frame. Preserve identity, pose, expression, gaze, body movement, interactions, lip-sync/performance and camera timing. Maintain temporal consistency: stable character design, outlines, colors and backgrounds. ZERO photorealism. ZERO live-action texture. ZERO 3D geometry. ZERO CGI. ZERO physically based materials. ZERO photographic skin/hair. ZERO realistic reflections. ZERO volumetric photographic lighting. Do not leave any original realistic region untouched. The COMPLETE FINAL FRAME must read as a single 2D animated illustration.'''
print(STYLE_PROMPT)


## 4. Connect to hosted ID-V2V and inspect its live API
The public Space can change its API or become unavailable, so this notebook inspects the current API instead of inventing an endpoint.


In [ ]:
SPACE='hugging-apps/id-v2v'
client=Client(SPACE)
api=client.view_api(all_endpoints=True,return_format='dict',print_info=False)
print(json.dumps(api,indent=2,default=str)[:30000])


## 5. One-click API discovery + real inference
Gradio exposes endpoint parameters through `view_api()`. This cell finds an endpoint containing video/image/prompt inputs, maps them by their labels, submits a real remote job, and returns the generated video. If no suitable endpoint is exposed, it stops honestly.


In [ ]:
def _all_named_endpoints(info):
    return (info or {}).get('named_endpoints', {}) if isinstance(info,dict) else {}

def _pick_endpoint(info):
    eps=_all_named_endpoints(info)
    ranked=[]
    for name,spec in eps.items():
        text=(name+' '+json.dumps(spec,default=str)).lower()
        score=sum(k in text for k in ['video','image','prompt','restyl','generate','infer'])
        ranked.append((score,name,spec))
    ranked.sort(key=lambda x:x[0],reverse=True)
    return ranked[0][1:] if ranked and ranked[0][0]>=2 else (None,None)

def _build_kwargs(spec):
    kwargs={}
    params=spec.get('parameters',[]) if isinstance(spec,dict) else []
    for p in params:
        label=str(p.get('label','')).lower()
        key=p.get('parameter_name') or p.get('name') or p.get('label')
        if not key: continue
        if any(k in label for k in ['video','source','input video']):
            kwargs[key]=handle_file(source_video)
        elif any(k in label for k in ['image','frame','keyframe','style']):
            kwargs[key]=handle_file('/content/2d_style_guide.png')
        elif 'prompt' in label:
            kwargs[key]=STYLE_PROMPT
    return kwargs

endpoint,spec=_pick_endpoint(api)
print('Selected endpoint:',endpoint)
if not endpoint:
    raise RuntimeError('No suitable public ID-V2V inference endpoint is exposed by the Space. Nothing was faked.')
kwargs=_build_kwargs(spec)
print('Mapped inputs:',list(kwargs.keys()))
if not any(isinstance(v,dict) and 'path' in v for v in kwargs.values()):
    raise RuntimeError('The live Space API did not expose a file input for the source video.')
if not any(isinstance(v,dict) and 'path' in v and str(v['path']).endswith('.png') for v in kwargs.values()):
    raise RuntimeError('The live Space API did not expose an image/keyframe input required by ID-V2V.')
if not any(isinstance(v,str) and '2D' in v for v in kwargs.values()):
    raise RuntimeError('The live Space API did not expose a text prompt input.')
result=client.predict(api_name=endpoint,**kwargs)
print('REMOTE RESULT:',result)


## 6. Locate generated video and restore original audio


In [ ]:
def _find_paths(x):
    out=[]
    if isinstance(x,str) and (x.endswith('.mp4') or x.endswith('.webm') or x.endswith('.mov')): out.append(x)
    elif isinstance(x,dict):
        for v in x.values(): out += _find_paths(v)
    elif isinstance(x,(list,tuple)):
        for v in x: out += _find_paths(v)
    return out
paths=_find_paths(result)
print('Video candidates:',paths)
if paths:
    GENERATED=paths[0]
    final='/content/2dify_full_2d.mp4'
    subprocess.run([ffmpeg,'-y','-i',GENERATED,'-i',source_video,'-map','0:v:0','-map','1:a?','-c:v','copy','-c:a','aac','-shortest',final],check=True)
    print('FINAL:',final)
else:
    print('The remote endpoint returned no video path. Inspect REMOTE RESULT above.')


In [ ]:
# Example only — replace ENDPOINT and argument order using the live view_api() output.
# result=client.predict(handle_file(source_video),handle_file('/content/2d_style_guide.png'),STYLE_PROMPT,api_name=ENDPOINT)
# print(result)


## 6. Restore original audio after successful generation


In [ ]:
GENERATED=''
if GENERATED and os.path.exists(GENERATED):
    final='/content/2dify_full_2d.mp4'
    subprocess.run([ffmpeg,'-y','-i',GENERATED,'-i',source_video,'-map','0:v:0','-map','1:a?','-c:v','copy','-c:a','aac','-shortest',final],check=True)
    print('FINAL:',final)
else:
    print('No generated file yet. Run real inference first.')


## Quality gate
The target is full-frame 2D. If any obvious realistic/3D region survives, treat the conversion as unsuccessful rather than labeling it 100% 2D. Generative video cannot mathematically guarantee every pixel, so visual QA remains necessary.
